# 01 - IPL Data Cleaning

Cleans the raw `matches.csv` and `deliveries.csv` files: standardises team and venue names across seasons, fixes dtypes, and fills missing values.

Standardisation rules live in [`src/data_cleaning.py`](../src/data_cleaning.py) so they can be reused by the other notebooks and unit tested independently of this notebook.

**Input:** `data/raw/matches.csv`, `data/raw/deliveries.csv`
**Output:** `data/processed/matches_c.csv`, `data/processed/deliveries_c.csv`

In [8]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_cleaning import clean_matches, clean_deliveries

## Load raw data

In [9]:
matches_df = pd.read_csv(r'C:\Users\HomePC\Downloads\ipl-data-analysis (2)\ipl-data-analysis\data\processed\matches_c.csv')

In [10]:
deliveries_df = pd.read_csv(r'C:\Users\HomePC\Downloads\ipl-data-analysis (2)\ipl-data-analysis\data\processed\deliveries_c.csv')

## Explore before cleaning
Quick look at shape, dtypes and the inconsistencies that need fixing (renamed franchises, duplicate venue spellings, missing values).

Match table cleaning Part

In [11]:
matches_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1095 entries, 0 to 1094
Data columns (total 20 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   id               1095 non-null   int64  
 1   season           1095 non-null   str    
 2   city             1095 non-null   str    
 3   date             1095 non-null   str    
 4   match_type       1095 non-null   str    
 5   player_of_match  1090 non-null   str    
 6   venue            1095 non-null   str    
 7   team1            1095 non-null   str    
 8   team2            1095 non-null   str    
 9   toss_winner      1095 non-null   str    
 10  toss_decision    1095 non-null   str    
 11  winner           1090 non-null   str    
 12  result           1095 non-null   str    
 13  result_margin    1076 non-null   float64
 14  target_runs      1092 non-null   float64
 15  target_overs     1092 non-null   float64
 16  super_over       1095 non-null   str    
 17  method           1095 non

In [12]:
matches_df['date']=pd.to_datetime(matches_df['date'])

In [13]:
matches_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1095 entries, 0 to 1094
Data columns (total 20 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   id               1095 non-null   int64         
 1   season           1095 non-null   str           
 2   city             1095 non-null   str           
 3   date             1095 non-null   datetime64[us]
 4   match_type       1095 non-null   str           
 5   player_of_match  1090 non-null   str           
 6   venue            1095 non-null   str           
 7   team1            1095 non-null   str           
 8   team2            1095 non-null   str           
 9   toss_winner      1095 non-null   str           
 10  toss_decision    1095 non-null   str           
 11  winner           1090 non-null   str           
 12  result           1095 non-null   str           
 13  result_margin    1076 non-null   float64       
 14  target_runs      1092 non-null   float64       
 15

In [14]:
matches_df.describe()

,id,date,result_margin,target_runs,target_overs
count,1.095000e+03,1095,1076.000000,1092.000000,1092.000000
mean,9.048283e+05,2016-06-25 11:39:36.986301,17.259294,165.684066,19.759341
min,3.359820e+05,2008-04-18 00:00:00,1.000000,43.000000,5.000000
25%,5.483315e+05,2012-04-21 00:00:00,6.000000,146.000000,20.000000
50%,9.809610e+05,2016-05-03 00:00:00,8.000000,166.000000,20.000000
75%,1.254062e+06,2021-04-13 12:00:00,20.000000,187.000000,20.000000
max,1.426312e+06,2024-05-26 00:00:00,146.000000,288.000000,20.000000
std,3.677402e+05,NaN,21.787444,33.427048,1.581108


In [15]:
matches_df.shape

(1095, 20)

In [16]:
#Match_df values count for team repeat values....
matches_df['team1'].value_counts()

team1
Royal Challengers Bengaluru    144
Chennai Super Kings            128
Delhi Capitals                 126
Punjab Kings                   123
Mumbai Indians                 123
Kolkata Knight Riders          121
Rajasthan Royals               101
Sunrisers Hyderabad             86
Deccan Chargers                 39
Pune Warriors                   23
Lucknow Super Giants            23
Gujarat Titans                  21
Gujarat Lions                   16
Rising Pune Supergiant          14
Kochi Tuskers Kerala             7
Name: count, dtype: int64

Match table team name replacment 

Correction off stadium names

In [17]:
matches_df['venue'].value_counts()

venue
Wankhede Stadium                                                         118
M Chinnaswamy Stadium                                                     94
Eden Gardens                                                              93
Rajiv Gandhi International Stadium                                        77
Punjab Cricket Association Stadium, Mohali                                61
Feroz Shah Kotla                                                          60
Sawai Mansingh Stadium                                                    57
MA Chidambaram Stadium                                                    57
Dubai International Cricket Stadium                                       46
Dr DY Patil Sports Academy                                                37
Maharashtra Cricket Association Stadium                                   35
Arun Jaitley Stadium                                                      30
Sheikh Zayed Stadium                                                  

In [18]:
matches_df['venue'].value_counts().sort_index(ascending=False)

venue
Zayed Cricket Stadium, Abu Dhabi                                           8
Wankhede Stadium                                                         118
Vidarbha Cricket Association Stadium, Jamtha                               3
SuperSport Park                                                           12
Subrata Roy Sahara Stadium                                                16
St George's Park                                                           7
Sheikh Zayed Stadium                                                      29
Sharjah Cricket Stadium                                                   28
Shaheed Veer Narayan Singh International Stadium                           6
Sawai Mansingh Stadium                                                    57
Saurashtra Cricket Association Stadium                                    10
Sardar Patel Stadium, Motera                                              12
Rajiv Gandhi International Stadium                                    

In [19]:
matches_df['team1'].value_counts()

team1
Royal Challengers Bengaluru    144
Chennai Super Kings            128
Delhi Capitals                 126
Punjab Kings                   123
Mumbai Indians                 123
Kolkata Knight Riders          121
Rajasthan Royals               101
Sunrisers Hyderabad             86
Deccan Chargers                 39
Pune Warriors                   23
Lucknow Super Giants            23
Gujarat Titans                  21
Gujarat Lions                   16
Rising Pune Supergiant          14
Kochi Tuskers Kerala             7
Name: count, dtype: int64

In [20]:
#Finding null values....
matches_df.isnull().sum()

id                  0
season              0
city                0
date                0
match_type          0
player_of_match     5
venue               0
team1               0
team2               0
toss_winner         0
toss_decision       0
winner              5
result              0
result_margin      19
target_runs         3
target_overs        3
super_over          0
method              0
umpire1             0
umpire2             0
dtype: int64

In [21]:
matches_df.isnull().sum()

id                  0
season              0
city                0
date                0
match_type          0
player_of_match     5
venue               0
team1               0
team2               0
toss_winner         0
toss_decision       0
winner              5
result              0
result_margin      19
target_runs         3
target_overs        3
super_over          0
method              0
umpire1             0
umpire2             0
dtype: int64

Delivery table cleaning Part


In [22]:
deliveries_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 260920 entries, 0 to 260919
Data columns (total 17 columns):
 #   Column            Non-Null Count   Dtype
---  ------            --------------   -----
 0   match_id          260920 non-null  int64
 1   inning            260920 non-null  int64
 2   batting_team      260920 non-null  str  
 3   bowling_team      260920 non-null  str  
 4   over              260920 non-null  int64
 5   ball              260920 non-null  int64
 6   batter            260920 non-null  str  
 7   bowler            260920 non-null  str  
 8   non_striker       260920 non-null  str  
 9   batsman_runs      260920 non-null  int64
 10  extra_runs        260920 non-null  int64
 11  total_runs        260920 non-null  int64
 12  extras_type       260920 non-null  str  
 13  is_wicket         260920 non-null  int64
 14  player_dismissed  260920 non-null  str  
 15  dismissal_kind    260920 non-null  str  
 16  fielder           260920 non-null  str  
dtypes: int64(8), str(9)
m

In [23]:
deliveries_df.describe()

,match_id,inning,over,ball,batsman_runs,extra_runs,total_runs,is_wicket
count,2.609200e+05,260920.000000,260920.000000,260920.000000,260920.000000,260920.000000,260920.000000,260920.000000
mean,9.070665e+05,1.483531,9.197677,3.624486,1.265001,0.067806,1.332807,0.049632
std,3.679913e+05,0.502643,5.683484,1.814920,1.639298,0.343265,1.626416,0.217184
min,3.359820e+05,1.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000
25%,5.483340e+05,1.000000,4.000000,2.000000,0.000000,0.000000,0.000000,0.000000
50%,9.809670e+05,1.000000,9.000000,4.000000,1.000000,0.000000,1.000000,0.000000
75%,1.254066e+06,2.000000,14.000000,5.000000,1.000000,0.000000,1.000000,0.000000
max,1.426312e+06,6.000000,19.000000,11.000000,6.000000,7.000000,7.000000,1.000000


In [24]:
deliveries_df.shape

(260920, 17)

Replace team name for data cleaning...

In [25]:
deliveries_df['batting_team'].value_counts()

batting_team
Mumbai Indians                 31437
Royal Challengers Bengaluru    30023
Delhi Capitals                 29732
Kolkata Knight Riders          29514
Punjab Kings                   29479
Chennai Super Kings            28651
Rajasthan Royals               26242
Sunrisers Hyderabad            21843
Deccan Chargers                 9034
Gujarat Titans                  5494
Pune Warriors                   5443
Lucknow Super Giants            5400
Gujarat Lions                   3566
Rising Pune Supergiant          3480
Kochi Tuskers Kerala            1582
Name: count, dtype: int64

In [26]:
#Finding null values..
deliveries_df.isnull().sum()

match_id            0
inning              0
batting_team        0
bowling_team        0
over                0
ball                0
batter              0
bowler              0
non_striker         0
batsman_runs        0
extra_runs          0
total_runs          0
extras_type         0
is_wicket           0
player_dismissed    0
dismissal_kind      0
fielder             0
dtype: int64

Fill null values......

In [27]:
deliveries_df.isnull().sum()

match_id            0
inning              0
batting_team        0
bowling_team        0
over                0
ball                0
batter              0
bowler              0
non_striker         0
batsman_runs        0
extra_runs          0
total_runs          0
extras_type         0
is_wicket           0
player_dismissed    0
dismissal_kind      0
fielder             0
dtype: int64

## Apply cleaning
All the team-name, venue-name and null-value fixes explored above are applied in one place via `src/data_cleaning.py`, instead of being repeated inline.

In [28]:
matches_df = clean_matches(matches_df)
deliveries_df = clean_deliveries(deliveries_df)

## Verify

In [29]:
matches_df.isnull().sum()

id                  0
season              0
city                0
date                0
match_type          0
player_of_match     5
venue               0
team1               0
team2               0
toss_winner         0
toss_decision       0
winner              5
result              0
result_margin      19
target_runs         3
target_overs        3
super_over          0
method              0
umpire1             0
umpire2             0
dtype: int64

In [30]:
deliveries_df.isnull().sum()

match_id            0
inning              0
batting_team        0
bowling_team        0
over                0
ball                0
batter              0
bowler              0
non_striker         0
batsman_runs        0
extra_runs          0
total_runs          0
extras_type         0
is_wicket           0
player_dismissed    0
dismissal_kind      0
fielder             0
dtype: int64

## Save cleaned data

In [31]:
matches_df.to_csv('../data/processed/matches_c.csv', index=False)
deliveries_df.to_csv('../data/processed/deliveries_c.csv', index=False)